In [1]:
# ============================================================
# D4 - Branch C: Deterministic Normalisation
# 0. Imports and frozen experimental configuration
# ============================================================
import json
import hashlib
import html
import platform
import re
import sys
import unicodedata

from collections import Counter
from datetime import datetime
from html.parser import HTMLParser
from pathlib import Path

import openpyxl
import pandas as pd
from google.colab import files

DOCUMENT_ID = "D4"
DOCUMENT_NAME = "Eurostat — LFS Metadata Excel — lfsa_esms"
BRANCH = "C"
BRANCH_NAME = "Deterministic normalisation"
PARENT_BRANCH = "B"

EXPECTED_SOURCE_FORMAT = ".xlsx"
EXPECTED_SOURCE_SHA256 = "040f1ace0392f8517fb99d4785b79cf3161cdd8e35f9c0e95bf243fe9baba5e9"

EXPECTED_SHEETS = ["Metadata", "Parameters", "Annexes"]
SOURCE_SHEET = "Metadata"
EXPECTED_SCOPE_ROWS = list(range(2, 10)) + list(range(12, 87))

EXPECTED_RECORD_COUNT = 83
EXPECTED_HEADER_RECORD_COUNT = 8
EXPECTED_CONCEPT_RECORD_COUNT = 75

EXPECTED_FIELDS = [
    "Section",
    "Concept Name",
    "Concept Value",
    "Publication Restricted"
]

ALLOWED_PUBLICATION_FLAGS = {"YES", "NO", None}

OUTPUT_DIR = Path("outputs_D4_branch_C")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Configured:", DOCUMENT_ID, BRANCH)
print("Parent branch:", PARENT_BRANCH)
print("Fixed extraction scope: Metadata rows 2–9 and 12–86")


Configured: D4 C
Parent branch: B
Fixed extraction scope: Metadata rows 2–9 and 12–86


In [2]:
# ============================================================
# 1. Upload original D4 XLSX and required Branch B artefacts
# ============================================================
# Upload exactly:
# 1) original D4 XLSX
# 2) D4_branch_B_structural_markdown.md
# 3) D4_branch_B_conversion_integrity.json

uploaded = files.upload()
names = list(uploaded.keys())

xlsx_files = [Path(f) for f in names if f.lower().endswith(".xlsx")]
md_files = [Path(f) for f in names if f.lower().endswith(".md")]
json_files = [Path(f) for f in names if f.lower().endswith(".json")]

if len(xlsx_files) != 1 or len(md_files) != 1 or len(json_files) != 1:
    raise ValueError(
        "Upload exactly one XLSX, one Branch B Markdown file, "
        "and one Branch B conversion-integrity JSON file."
    )

SOURCE_PATH = xlsx_files[0]
BRANCH_B_REPRESENTATION_PATH = md_files[0]
BRANCH_B_CHECK_PATH = json_files[0]

print("Source:", SOURCE_PATH.name)
print("Branch B representation:", BRANCH_B_REPRESENTATION_PATH.name)
print("Branch B integrity:", BRANCH_B_CHECK_PATH.name)


Saving D4_branch_B_conversion_integrity.json to D4_branch_B_conversion_integrity.json
Saving D4_branch_B_structural_markdown.md to D4_branch_B_structural_markdown.md
Saving D4 - Eurostat – LFS Metadata Excel - lfsa_esms.xlsx to D4 - Eurostat – LFS Metadata Excel - lfsa_esms.xlsx
Source: D4 - Eurostat – LFS Metadata Excel - lfsa_esms.xlsx
Branch B representation: D4_branch_B_structural_markdown.md
Branch B integrity: D4_branch_B_conversion_integrity.json


In [3]:
# ============================================================
# 2. Verify frozen source identity and Branch B parent integrity
# ============================================================
def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def sha256_text(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

if SOURCE_PATH.suffix.lower() != EXPECTED_SOURCE_FORMAT:
    raise ValueError("Unexpected D4 source format.")

SOURCE_SHA256 = sha256_file(SOURCE_PATH)
SOURCE_HASH_MATCH = SOURCE_SHA256 == EXPECTED_SOURCE_SHA256

if not SOURCE_HASH_MATCH:
    raise ValueError("Uploaded workbook does not match the frozen D4 source identity.")

with open(BRANCH_B_CHECK_PATH, "r", encoding="utf-8") as f:
    branch_b_check = json.load(f)

if branch_b_check.get("document_id") != DOCUMENT_ID:
    raise ValueError("Branch B integrity artefact belongs to another document.")
if branch_b_check.get("branch") != "B":
    raise ValueError("Uploaded integrity artefact is not from Branch B.")
if branch_b_check.get("source_sha256") != SOURCE_SHA256:
    raise ValueError("Branch B parent representation was created from a different source.")
if not branch_b_check.get("conversion_integrity_passed", False):
    raise ValueError("Branch B parent representation did not pass conversion integrity.")

SOURCE_B_MARKDOWN = BRANCH_B_REPRESENTATION_PATH.read_text(encoding="utf-8")
if not SOURCE_B_MARKDOWN.strip():
    raise ValueError("Uploaded Branch B Markdown is empty.")

SOURCE_B_SHA256 = sha256_text(SOURCE_B_MARKDOWN)

print("Frozen source identity verified.")
print("Branch B conversion integrity verified.")
print("Branch B representation SHA-256:", SOURCE_B_SHA256)


Frozen source identity verified.
Branch B conversion integrity verified.
Branch B representation SHA-256: 99a8dfdbb5afad033e042b4ba42fd8dbc82a166f0e7159d4aa91d0c1bcdbfe40


In [4]:
# ============================================================
# 3. Reproduce the exact Branch B structural representation
# ============================================================
workbook = openpyxl.load_workbook(SOURCE_PATH, data_only=False)
sheet_names = workbook.sheetnames

if sheet_names != EXPECTED_SHEETS:
    raise ValueError(
        f"Unexpected worksheet structure. Expected {EXPECTED_SHEETS}; observed {sheet_names}"
    )

metadata_ws = workbook[SOURCE_SHEET]
TOP_LEVEL_RE = re.compile(r"^(\d+)\.\s+")

def preserve_cell(value):
    if value is None:
        return None
    if isinstance(value, str) and value == "":
        return None
    return value

target_records = []
current_section = None

for row in range(2, 10):
    target_records.append({
        "Source Row": row,
        "Section": "Header",
        "Concept Name": preserve_cell(metadata_ws.cell(row, 1).value),
        "Concept Value": preserve_cell(metadata_ws.cell(row, 2).value),
        "Publication Restricted": preserve_cell(metadata_ws.cell(row, 3).value)
    })

for row in range(12, 87):
    concept_name = preserve_cell(metadata_ws.cell(row, 1).value)
    if isinstance(concept_name, str) and TOP_LEVEL_RE.match(concept_name):
        current_section = concept_name

    target_records.append({
        "Source Row": row,
        "Section": current_section,
        "Concept Name": concept_name,
        "Concept Value": preserve_cell(metadata_ws.cell(row, 2).value),
        "Publication Restricted": preserve_cell(metadata_ws.cell(row, 3).value)
    })

target_df = pd.DataFrame(target_records)

if len(target_df) != EXPECTED_RECORD_COUNT:
    raise ValueError("Could not reproduce the fixed D4 target-scope records.")

def markdown_inline(value):
    if value is None:
        return "`null`"
    text = str(value).replace("`", "\\`")
    return f"`{text}`"

def fenced_source_value(value):
    if value is None:
        return "```text\nnull\n```"
    return "```text\n" + str(value) + "\n```"

markdown_lines = [
    "# D4 — Eurostat LFS Metadata",
    "",
    "> Structural conversion of the complete original XLSX workbook.",
    "> The fixed extraction task remains restricted to Metadata rows 2–9 and 12–86.",
    ""
]

for sheet_name in sheet_names:
    ws = workbook[sheet_name]
    markdown_lines += [f"## Worksheet: {sheet_name}", ""]

    for row in range(1, ws.max_row + 1):
        values = [ws.cell(row, col).value for col in range(1, ws.max_column + 1)]
        if all(v is None for v in values):
            continue

        markdown_lines += [f"### Source Row {row}", ""]

        for col, value in enumerate(values, start=1):
            rendered = "`null`" if value is None else fenced_source_value(value)
            markdown_lines += [f"**Column {col}**", "", rendered, ""]

    if sheet_name == SOURCE_SHEET:
        markdown_lines += ["## Target-scope structural interpretation", ""]

        for _, record in target_df.iterrows():
            markdown_lines += [
                f"### Target Record — Source Row {int(record['Source Row'])}",
                "",
                f"- Section: {markdown_inline(record['Section'])}",
                f"- Concept Name: {markdown_inline(record['Concept Name'])}",
                "- Concept Value:",
                fenced_source_value(record["Concept Value"]),
                "- Publication Restricted: "
                f"{markdown_inline(record['Publication Restricted'])}",
                ""
            ]

REPRODUCED_BRANCH_B_MARKDOWN = "\n".join(markdown_lines).rstrip() + "\n"
REPRODUCED_B_SHA256 = sha256_text(REPRODUCED_BRANCH_B_MARKDOWN)

print("Reproduced Branch B SHA-256:", REPRODUCED_B_SHA256)


Reproduced Branch B SHA-256: 99a8dfdbb5afad033e042b4ba42fd8dbc82a166f0e7159d4aa91d0c1bcdbfe40


/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [5]:
# ============================================================
# 4. Verify exact Branch B parent equivalence
# ============================================================
PARENT_EQUIVALENCE_PASSED = (
    REPRODUCED_BRANCH_B_MARKDOWN == SOURCE_B_MARKDOWN
)

parent_check = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,
    "parent_branch": PARENT_BRANCH,
    "source_sha256": SOURCE_SHA256,
    "branch_B_conversion_integrity_passed":
        bool(branch_b_check.get("conversion_integrity_passed", False)),
    "uploaded_branch_B_sha256": SOURCE_B_SHA256,
    "reproduced_branch_B_sha256": REPRODUCED_B_SHA256,
    "branch_B_representation_exactly_reproduced": PARENT_EQUIVALENCE_PASSED,
    "target_record_count": int(len(target_df)),
    "parent_equivalence_passed": PARENT_EQUIVALENCE_PASSED
}

PARENT_CHECK_PATH = OUTPUT_DIR / "D4_branch_C_parent_B_equivalence_check.json"
PARENT_CHECK_PATH.write_text(
    json.dumps(parent_check, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print(json.dumps(parent_check, indent=2, ensure_ascii=False))

if not PARENT_EQUIVALENCE_PASSED:
    raise ValueError(
        "The uploaded Branch B representation does not exactly match "
        "the representation reproduced from the frozen D4 workbook."
    )


{
  "document_id": "D4",
  "branch": "C",
  "parent_branch": "B",
  "source_sha256": "040f1ace0392f8517fb99d4785b79cf3161cdd8e35f9c0e95bf243fe9baba5e9",
  "branch_B_conversion_integrity_passed": true,
  "uploaded_branch_B_sha256": "99a8dfdbb5afad033e042b4ba42fd8dbc82a166f0e7159d4aa91d0c1bcdbfe40",
  "reproduced_branch_B_sha256": "99a8dfdbb5afad033e042b4ba42fd8dbc82a166f0e7159d4aa91d0c1bcdbfe40",
  "branch_B_representation_exactly_reproduced": true,
  "target_record_count": 83,
  "parent_equivalence_passed": true
}


In [6]:
# ============================================================
# 5. Define deterministic D4 Branch C normalisation
# ============================================================
UNICODE_SPACE_CHARACTERS = [
    "\u00a0", "\u1680", "\u2000", "\u2001", "\u2002",
    "\u2003", "\u2004", "\u2005", "\u2006", "\u2007",
    "\u2008", "\u2009", "\u200a", "\u202f", "\u205f", "\u3000"
]

APOSTROPHE_REPLACEMENTS = {
    "’": "'", "‘": "'", "‛": "'", "´": "'", "`": "'"
}

DASH_REPLACEMENTS = {
    "–": "-", "—": "-", "−": "-", "‐": "-"
}

class MetadataHTMLNormaliser(HTMLParser):
    BLOCK_TAGS = {
        "p", "div", "li", "ul", "ol", "table", "tr", "td", "th",
        "section", "article", "h1", "h2", "h3", "h4", "h5", "h6"
    }

    def __init__(self):
        super().__init__(convert_charrefs=True)
        self.parts = []
        self.active_link = None
        self.active_link_text = []

    def add_boundary(self):
        if self.parts and self.parts[-1] != "\n":
            self.parts.append("\n")

    def handle_starttag(self, tag, attrs):
        tag = tag.lower()
        attributes = dict(attrs)

        if tag in self.BLOCK_TAGS or tag == "br":
            self.add_boundary()

        if tag == "a":
            self.active_link = attributes.get("href")
            self.active_link_text = []

    def handle_endtag(self, tag):
        tag = tag.lower()

        if tag == "a":
            if self.active_link:
                label = "".join(self.active_link_text).strip()
                if label:
                    self.parts.append(f" [{self.active_link}]")
                else:
                    self.parts.append(self.active_link)

            self.active_link = None
            self.active_link_text = []

        if tag in self.BLOCK_TAGS:
            self.add_boundary()

    def handle_data(self, data):
        self.parts.append(data)
        if self.active_link is not None:
            self.active_link_text.append(data)

    def get_text(self):
        return "".join(self.parts)

def normalise_plain_text(value):
    if value is None:
        return None

    text = str(value)
    if text == "null":
        return None

    text = unicodedata.normalize("NFKC", text)

    for character in UNICODE_SPACE_CHARACTERS:
        text = text.replace(character, " ")

    for source, target in APOSTROPHE_REPLACEMENTS.items():
        text = text.replace(source, target)

    for source, target in DASH_REPLACEMENTS.items():
        text = text.replace(source, target)

    text = text.replace("\r\n", "\n").replace("\r", "\n")

    parser = MetadataHTMLNormaliser()

    try:
        parser.feed(text)
        parser.close()
        text = parser.get_text()
    except Exception:
        text = html.unescape(text)

    text = html.unescape(text)

    normalised_lines = []

    for line in text.splitlines():
        line = re.sub(r"[ \t\f\v]+", " ", line).strip()
        if line:
            normalised_lines.append(line)

    if not normalised_lines:
        return None

    return "\n".join(normalised_lines)

def normalise_inline_text(value):
    if value is None:
        return None

    text = unicodedata.normalize("NFKC", str(value))

    for character in UNICODE_SPACE_CHARACTERS:
        text = text.replace(character, " ")

    for source, target in APOSTROPHE_REPLACEMENTS.items():
        text = text.replace(source, target)

    for source, target in DASH_REPLACEMENTS.items():
        text = text.replace(source, target)

    text = re.sub(r"[ \t\f\v]+", " ", text).strip()
    return text if text else None

HTML_TAG_PATTERN = re.compile(r"<[^>]+>")
HTML_ENTITY_PATTERN = re.compile(r"&(?:#\d+|#x[0-9A-Fa-f]+|[A-Za-z]+);")
HREF_PATTERN = re.compile("href\\s*=\\s*[\"']([^\"']+)[\"']", flags=re.IGNORECASE)


In [7]:
# ============================================================
# 6. Apply Branch C normalisation to the Branch B representation
# ============================================================
FENCED_BLOCK_PATTERN = re.compile(
    r"```text\n(.*?)\n```",
    flags=re.DOTALL
)

def normalise_fenced_match(match):
    source_content = match.group(1)
    normalised_content = normalise_plain_text(source_content)
    rendered = "null" if normalised_content is None else normalised_content
    return "```text\n" + rendered + "\n```"

NORMALISED_MARKDOWN = FENCED_BLOCK_PATTERN.sub(
    normalise_fenced_match,
    SOURCE_B_MARKDOWN
)

def normalise_inline_field(match):
    prefix = match.group(1)
    value = match.group(2)

    if value == "null":
        return prefix + "`null`"

    normalised = normalise_inline_text(
        value.replace("\\`", "`")
    )

    if normalised is None:
        return prefix + "`null`"

    normalised = normalised.replace("`", "\\`")
    return prefix + f"`{normalised}`"

INLINE_FIELD_PATTERN = re.compile(
    r"(^- (?:Section|Concept Name|Publication Restricted): )`(.*?)`$",
    flags=re.MULTILINE
)

NORMALISED_MARKDOWN = INLINE_FIELD_PATTERN.sub(
    normalise_inline_field,
    NORMALISED_MARKDOWN
)

NORMALISED_MARKDOWN = (
    "\n".join(
        line.rstrip()
        for line in NORMALISED_MARKDOWN.splitlines()
    ).rstrip()
    + "\n"
)

if not NORMALISED_MARKDOWN.strip():
    raise ValueError("Branch C normalisation produced an empty representation.")

print("Branch B characters:", len(SOURCE_B_MARKDOWN))
print("Branch C characters:", len(NORMALISED_MARKDOWN))


Branch B characters: 72080
Branch C characters: 64414


In [10]:
# ============================================================
# 7. Verify Branch C normalisation integrity
# ============================================================
#
# IMPORTANT D4 PRINCIPLE
# ----------------------
# Branch C deliberately removes HTML markup, decodes HTML entities,
# standardises Unicode/whitespace and converts content with no
# remaining visible information to null.
#
# Therefore, the literal number of null fenced blocks is NOT
# expected to remain identical to Branch B.
#
# Example:
#
# Branch B:
#     ```text
#     <p>&nbsp;</p>
#     ```
#
# Branch C:
#     ```text
#     null
#     ```
#
# This is a valid deterministic normalisation, not information loss.
#
# Integrity is therefore checked by verifying that every Branch C
# fenced block is exactly equal to the output of the predefined
# normalise_plain_text() function applied to the corresponding
# Branch B block.
# ============================================================


# ------------------------------------------------------------
# A. Complete workbook structure must remain represented
# ------------------------------------------------------------

worksheet_marker_checks = {
    sheet: (
        f"## Worksheet: {sheet}"
        in NORMALISED_MARKDOWN
    )
    for sheet in EXPECTED_SHEETS
}

all_worksheets_preserved = all(
    worksheet_marker_checks.values()
)


# ------------------------------------------------------------
# B. Source-row structure must remain in the same sequence
# ------------------------------------------------------------

SOURCE_ROW_PATTERN = re.compile(
    r"^### Source Row (\d+)$",
    flags=re.MULTILINE
)

parent_source_rows = (
    SOURCE_ROW_PATTERN.findall(
        SOURCE_B_MARKDOWN
    )
)

branch_c_source_rows = (
    SOURCE_ROW_PATTERN.findall(
        NORMALISED_MARKDOWN
    )
)

source_row_sequence_preserved = (
    parent_source_rows
    == branch_c_source_rows
)


# ------------------------------------------------------------
# C. Fixed Stage 1 target scope must remain unchanged
# ------------------------------------------------------------

TARGET_ROW_PATTERN = re.compile(
    r"^### Target Record — Source Row (\d+)$",
    flags=re.MULTILINE
)

parent_target_rows = [
    int(value)
    for value
    in TARGET_ROW_PATTERN.findall(
        SOURCE_B_MARKDOWN
    )
]

branch_c_target_rows = [
    int(value)
    for value
    in TARGET_ROW_PATTERN.findall(
        NORMALISED_MARKDOWN
    )
]

target_scope_sequence_preserved = bool(
    parent_target_rows
    == EXPECTED_SCOPE_ROWS
    and branch_c_target_rows
    == EXPECTED_SCOPE_ROWS
)

target_record_count_preserved = (
    len(branch_c_target_rows)
    == EXPECTED_RECORD_COUNT
)


# ------------------------------------------------------------
# D. Fenced source-value blocks must remain one-to-one
# ------------------------------------------------------------

parent_fenced_blocks = (
    FENCED_BLOCK_PATTERN.findall(
        SOURCE_B_MARKDOWN
    )
)

branch_c_fenced_blocks = (
    FENCED_BLOCK_PATTERN.findall(
        NORMALISED_MARKDOWN
    )
)

fenced_block_count_preserved = (
    len(parent_fenced_blocks)
    == len(branch_c_fenced_blocks)
)


# ------------------------------------------------------------
# E. Determine the EXPECTED Branch C value for every Branch B
#    fenced source block
# ------------------------------------------------------------

expected_branch_c_blocks = []

for parent_block in parent_fenced_blocks:

    normalised_value = normalise_plain_text(
        parent_block
    )

    expected_rendered_value = (
        "null"
        if normalised_value is None
        else normalised_value
    )

    expected_branch_c_blocks.append(
        expected_rendered_value
    )


# ------------------------------------------------------------
# F. Verify every actual Branch C block against the expected
#    deterministic transformation
# ------------------------------------------------------------

block_normalisation_mismatches = []

if fenced_block_count_preserved:

    for block_index, (
        expected_block,
        observed_block
    ) in enumerate(
        zip(
            expected_branch_c_blocks,
            branch_c_fenced_blocks
        )
    ):

        if expected_block != observed_block:

            block_normalisation_mismatches.append({
                "block_index":
                    block_index,

                "expected_after_normalisation":
                    expected_block,

                "observed_branch_C":
                    observed_block
            })


deterministic_block_transformation_verified = (
    len(
        block_normalisation_mismatches
    )
    == 0
)


# ------------------------------------------------------------
# G. Null-result accounting
# ------------------------------------------------------------
#
# The important comparison is NOT:
#
#     Branch B nulls == Branch C nulls
#
# Instead:
#
#     Expected nulls after applying Branch C rules
#     ==
#     Actual Branch C nulls
#
# ------------------------------------------------------------

parent_literal_null_count = sum(
    block == "null"
    for block
    in parent_fenced_blocks
)

expected_branch_c_null_count = sum(
    block == "null"
    for block
    in expected_branch_c_blocks
)

observed_branch_c_null_count = sum(
    block == "null"
    for block
    in branch_c_fenced_blocks
)

normalisation_generated_null_count = (
    expected_branch_c_null_count
    - parent_literal_null_count
)

expected_null_structure_verified = (
    observed_branch_c_null_count
    == expected_branch_c_null_count
)


# ------------------------------------------------------------
# H. Identify which Branch B blocks legitimately became null
# ------------------------------------------------------------

blocks_normalised_to_null = []

for block_index, (
    parent_block,
    expected_block
) in enumerate(
    zip(
        parent_fenced_blocks,
        expected_branch_c_blocks
    )
):

    if (
        parent_block != "null"
        and expected_block == "null"
    ):

        blocks_normalised_to_null.append({
            "block_index":
                block_index,

            "source_preview":
                parent_block[:300]
        })


# ------------------------------------------------------------
# I. HTML tags/entities must no longer remain
# ------------------------------------------------------------

normalised_values_with_html_tags = sum(
    bool(
        HTML_TAG_PATTERN.search(
            block
        )
    )
    for block
    in branch_c_fenced_blocks
    if block != "null"
)

normalised_values_with_html_entities = sum(
    bool(
        HTML_ENTITY_PATTERN.search(
            block
        )
    )
    for block
    in branch_c_fenced_blocks
    if block != "null"
)


# ------------------------------------------------------------
# J. Hyperlink targets from source HTML must remain represented
# ------------------------------------------------------------

parent_hrefs = []

for block in parent_fenced_blocks:

    parent_hrefs.extend(
        HREF_PATTERN.findall(
            block
        )
    )


parent_href_counter = Counter(
    parent_hrefs
)


missing_href_targets = {

    href: expected_count

    for href, expected_count
    in parent_href_counter.items()

    if NORMALISED_MARKDOWN.count(
        f"[{href}]"
    )
    < expected_count
}


hyperlink_targets_preserved = (
    len(missing_href_targets)
    == 0
)


# ------------------------------------------------------------
# K. Final Branch C integrity decision
# ------------------------------------------------------------

normalisation_integrity_passed = bool(

    PARENT_EQUIVALENCE_PASSED

    and all_worksheets_preserved

    and source_row_sequence_preserved

    and target_scope_sequence_preserved

    and target_record_count_preserved

    and fenced_block_count_preserved

    and deterministic_block_transformation_verified

    and expected_null_structure_verified

    and normalised_values_with_html_tags
    == 0

    and normalised_values_with_html_entities
    == 0

    and hyperlink_targets_preserved
)


# ------------------------------------------------------------
# L. Build detailed integrity report
# ------------------------------------------------------------

normalisation_check = {

    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "parent_branch":
        PARENT_BRANCH,


    # --------------------------------------------------------
    # Parent Branch B
    # --------------------------------------------------------

    "parent_equivalence_passed":
        PARENT_EQUIVALENCE_PASSED,


    # --------------------------------------------------------
    # Workbook structure
    # --------------------------------------------------------

    "worksheet_marker_checks":
        worksheet_marker_checks,

    "all_worksheets_preserved":
        all_worksheets_preserved,


    "parent_source_row_marker_count":
        len(
            parent_source_rows
        ),

    "branch_C_source_row_marker_count":
        len(
            branch_c_source_rows
        ),

    "source_row_sequence_preserved":
        source_row_sequence_preserved,


    # --------------------------------------------------------
    # Fixed extraction scope
    # --------------------------------------------------------

    "expected_target_rows":
        EXPECTED_SCOPE_ROWS,

    "parent_target_rows":
        parent_target_rows,

    "branch_C_target_rows":
        branch_c_target_rows,

    "target_scope_sequence_preserved":
        target_scope_sequence_preserved,

    "target_record_count_preserved":
        target_record_count_preserved,


    # --------------------------------------------------------
    # Fenced source-value structure
    # --------------------------------------------------------

    "parent_fenced_block_count":
        len(
            parent_fenced_blocks
        ),

    "branch_C_fenced_block_count":
        len(
            branch_c_fenced_blocks
        ),

    "fenced_block_count_preserved":
        fenced_block_count_preserved,


    # --------------------------------------------------------
    # Deterministic transformation verification
    # --------------------------------------------------------

    "block_normalisation_mismatch_count":
        len(
            block_normalisation_mismatches
        ),

    "block_normalisation_mismatches":
        block_normalisation_mismatches[
            :20
        ],

    "deterministic_block_transformation_verified":
        deterministic_block_transformation_verified,


    # --------------------------------------------------------
    # Null accounting
    # --------------------------------------------------------

    "parent_literal_null_count":
        int(
            parent_literal_null_count
        ),

    "expected_branch_C_null_count":
        int(
            expected_branch_c_null_count
        ),

    "observed_branch_C_null_count":
        int(
            observed_branch_c_null_count
        ),

    "normalisation_generated_null_count":
        int(
            normalisation_generated_null_count
        ),

    "expected_null_structure_verified":
        expected_null_structure_verified,

    "blocks_normalised_to_null_count":
        len(
            blocks_normalised_to_null
        ),

    "blocks_normalised_to_null_preview":
        blocks_normalised_to_null[
            :20
        ],


    # --------------------------------------------------------
    # HTML / entity normalisation
    # --------------------------------------------------------

    "normalised_values_with_html_tags":
        int(
            normalised_values_with_html_tags
        ),

    "normalised_values_with_html_entities":
        int(
            normalised_values_with_html_entities
        ),


    # --------------------------------------------------------
    # Hyperlink preservation
    # --------------------------------------------------------

    "source_html_href_count":
        int(
            sum(
                parent_href_counter.values()
            )
        ),

    "missing_href_targets":
        missing_href_targets,

    "hyperlink_targets_preserved":
        hyperlink_targets_preserved,


    # --------------------------------------------------------
    # Representation preservation
    # --------------------------------------------------------

    "complete_workbook_retained":
        True,

    "worksheet_filtering_applied":
        False,

    "out_of_scope_content_retained":
        True,


    # --------------------------------------------------------
    # Applied Branch C transformations
    # --------------------------------------------------------

    "unicode_nfkc_normalisation_applied":
        True,

    "unicode_space_standardisation_applied":
        True,

    "apostrophe_standardisation_applied":
        True,

    "dash_standardisation_applied":
        True,

    "html_cleaning_applied":
        True,

    "html_entities_decoded":
        True,

    "hyperlinks_preserved_as_text_and_url":
        True,

    "horizontal_whitespace_normalisation_applied":
        True,


    # --------------------------------------------------------
    # Explicitly NOT applied
    # --------------------------------------------------------

    "semantic_harmonisation_applied":
        False,

    "semantic_rewriting_applied":
        False,

    "manual_reconstruction_applied":
        False,

    "manual_correction_applied":
        False,

    "value_calculation_applied":
        False,

    "reference_values_used_for_transformation":
        False,


    # --------------------------------------------------------
    # Final integrity result
    # --------------------------------------------------------

    "normalisation_integrity_passed":
        normalisation_integrity_passed
}


# ------------------------------------------------------------
# M. Save integrity report
# ------------------------------------------------------------

NORMALISATION_CHECK_PATH = (
    OUTPUT_DIR
    / "D4_branch_C_normalisation_check.json"
)

NORMALISATION_CHECK_PATH.write_text(
    json.dumps(
        normalisation_check,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)


# ------------------------------------------------------------
# N. Display
# ------------------------------------------------------------

print(
    json.dumps(
        normalisation_check,
        indent=2,
        ensure_ascii=False
    )
)


# ------------------------------------------------------------
# O. Stop only for a genuine integrity failure
# ------------------------------------------------------------

if not normalisation_integrity_passed:

    raise ValueError(
        "D4 Branch C normalisation-integrity checks failed. "
        "Inspect workbook structure, deterministic block transformation, "
        "expected-null accounting, HTML/entity cleanup and hyperlink "
        "preservation."
    )

{
  "document_id": "D4",
  "branch": "C",
  "parent_branch": "B",
  "parent_equivalence_passed": true,
  "worksheet_marker_checks": {
    "Metadata": true,
    "Parameters": true,
    "Annexes": true
  },
  "all_worksheets_preserved": true,
  "parent_source_row_marker_count": 172,
  "branch_C_source_row_marker_count": 172,
  "source_row_sequence_preserved": true,
  "expected_target_rows": [
    2,
    3,
    4,
    5,
    6,
    7,
    8,
    9,
    12,
    13,
    14,
    15,
    16,
    17,
    18,
    19,
    20,
    21,
    22,
    23,
    24,
    25,
    26,
    27,
    28,
    29,
    30,
    31,
    32,
    33,
    34,
    35,
    36,
    37,
    38,
    39,
    40,
    41,
    42,
    43,
    44,
    45,
    46,
    47,
    48,
    49,
    50,
    51,
    52,
    53,
    54,
    55,
    56,
    57,
    58,
    59,
    60,
    61,
    62,
    63,
    64,
    65,
    66,
    67,
    68,
    69,
    70,
    71,
    72,
    73,
    74,
    75,
    76,
    77,
    78,
    79,
    80

In [11]:
# ============================================================
# 8. Save Branch C representation
# ============================================================
REPRESENTATION_PATH = (
    OUTPUT_DIR / "D4_branch_C_normalised_markdown.md"
)

REPRESENTATION_PATH.write_text(
    NORMALISED_MARKDOWN,
    encoding="utf-8"
)

REPRESENTATION_SHA256 = sha256_file(
    REPRESENTATION_PATH
)

print("Saved:", REPRESENTATION_PATH.name)
print("Representation SHA-256:", REPRESENTATION_SHA256)


Saved: D4_branch_C_normalised_markdown.md
Representation SHA-256: 465f1e7d420803d05bc3a501f6a37968dd2f3eb6df58323138c527f23447956b


In [12]:
# ============================================================
# 9. Define fixed output schema
# ============================================================
EXPECTED_OUTPUT_STRUCTURE = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,
    "records": [
        {
            "Section": None,
            "Concept Name": None,
            "Concept Value": None,
            "Publication Restricted": None
        }
    ]
}

print(json.dumps(
    EXPECTED_OUTPUT_STRUCTURE,
    indent=2,
    ensure_ascii=False
))


{
  "document_id": "D4",
  "branch": "C",
  "records": [
    {
      "Section": null,
      "Concept Name": null,
      "Concept Value": null,
      "Publication Restricted": null
    }
  ]
}


In [13]:
# ============================================================
# 10. Operationalise the fixed Stage 1 extraction task
# ============================================================
# Same substantive scope/schema as Branch B.
# The expected answer count is NOT disclosed.

EXTRACTION_TASK = """
You are an information extraction assistant.

Extract every metadata record represented within the defined scope of
the Metadata worksheet in the attached deterministically normalised
Markdown document.

For every included record, extract:

- Section
- Concept Name
- Concept Value
- Publication Restricted

Scope rules:

- Treat the attached deterministically normalised Markdown document as
  the only source of information.
- Use only the content corresponding to the Metadata worksheet.
- Include the workbook-header metadata records corresponding to source
  rows 2–9.
- Include every metadata concept record corresponding to source rows
  12–86.
- Include top-level metadata section rows even when their Concept Value
  is empty.
- Exclude source row 1, labelled "Header".
- Exclude source row 10, labelled "Concepts".
- Exclude source row 11, containing the source column headings.
- Do not extract records from the Parameters worksheet.
- Do not extract records from the Annexes worksheet.

Extraction rules:

- Preserve Concept Name exactly as represented in the normalised representation.
- Preserve Concept Value exactly as represented inside each normalised
  fenced text block, including punctuation, bracketed hyperlinks and
  internal line breaks.
- Do not reconstruct or reintroduce removed HTML tags or attributes.
- Do not alter, shorten, follow or rewrite hyperlinks.
- Preserve YES and NO publication-restriction flags exactly.
- Use null when a Concept Value or Publication Restricted value is
  explicitly represented as null.
- Assign workbook-header records from source rows 2–9 to Section "Header".
- For numbered metadata concepts, use the complete top-level numbered
  section heading as Section.
- Ignore source-row information because it is representation metadata
  and is not a requested extraction field.
- Do not follow hyperlinks.
- Do not infer missing values.
- Do not calculate, summarise, paraphrase, translate, harmonise or
  correct source content.
- Do not use external knowledge.
- Return one record for every included source row.
- Verify that only the fixed Metadata worksheet scope has been processed.
- Verify that every represented record within that scope has been processed.
- Return only valid JSON.
- Do not include Markdown fences, explanations or commentary.
- Keep the exact field names defined in the schema.
""".strip()

FULL_PROMPT = f"""
{EXTRACTION_TASK}

Expected JSON schema:
{json.dumps(
    EXPECTED_OUTPUT_STRUCTURE,
    indent=2,
    ensure_ascii=False
)}

The complete deterministically normalised representation of the
original XLSX workbook is attached.

The extraction scope remains restricted to Metadata source rows 2–9
and 12–86. Other workbook content is retained only to preserve the
same overall source exposure as Branches A and B and must not be extracted.

Return only the JSON object.
""".strip()

PROMPT_PATH = OUTPUT_DIR / "D4_branch_C_prompt.txt"
PROMPT_PATH.write_text(FULL_PROMPT, encoding="utf-8")
PROMPT_SHA256 = sha256_file(PROMPT_PATH)

print(FULL_PROMPT)
print("Prompt SHA-256:", PROMPT_SHA256)


You are an information extraction assistant.

Extract every metadata record represented within the defined scope of
the Metadata worksheet in the attached deterministically normalised
Markdown document.

For every included record, extract:

- Section
- Concept Name
- Concept Value
- Publication Restricted

Scope rules:

- Treat the attached deterministically normalised Markdown document as
  the only source of information.
- Use only the content corresponding to the Metadata worksheet.
- Include the workbook-header metadata records corresponding to source
  rows 2–9.
- Include every metadata concept record corresponding to source rows
  12–86.
- Include top-level metadata section rows even when their Concept Value
  is empty.
- Exclude source row 1, labelled "Header".
- Exclude source row 10, labelled "Concepts".
- Exclude source row 11, containing the source column headings.
- Do not extract records from the Parameters worksheet.
- Do not extract records from the Annexes worksheet.

E

In [14]:
# ============================================================
# 11. Create representation and experiment metadata
# ============================================================
REPRESENTATION_METADATA = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,
    "parent_branch": PARENT_BRANCH,
    "source_file": SOURCE_PATH.name,
    "source_format": SOURCE_PATH.suffix.lower(),
    "source_sha256": SOURCE_SHA256,
    "parent_B_representation_file": BRANCH_B_REPRESENTATION_PATH.name,
    "parent_B_representation_sha256": SOURCE_B_SHA256,
    "parent_B_equivalence_passed": PARENT_EQUIVALENCE_PASSED,
    "representation_type":
        "Complete Branch B structural Markdown with deterministic normalisation",
    "representation_file": REPRESENTATION_PATH.name,
    "representation_sha256": REPRESENTATION_SHA256,
    "complete_workbook_retained": True,
    "worksheets_retained": EXPECTED_SHEETS,
    "worksheet_filtering_applied": False,
    "out_of_scope_content_retained": True,
    "fixed_extraction_scope": "Metadata rows 2–9 and 12–86",
    "structural_conversion_inherited_from_branch_B": True,
    "normalisation_applied": True,
    "normalisation_operations": [
        "Unicode NFKC normalisation",
        "Unicode-space standardisation",
        "apostrophe standardisation",
        "dash standardisation",
        "HTML-like tag removal inside fenced source-value blocks",
        "HTML entity decoding",
        "paragraph/block boundary preservation as line breaks",
        "hyperlink preservation as Label [URL]",
        "horizontal whitespace normalisation",
        "inline target-label Unicode/spacing normalisation"
    ],
    "semantic_harmonisation_applied": False,
    "semantic_rewriting_applied": False,
    "manual_reconstruction_applied": False,
    "manual_correction_applied": False,
    "reference_values_used_for_transformation": False,
    "normalisation_integrity_passed":
        normalisation_check["normalisation_integrity_passed"]
}

REP_METADATA_PATH = (
    OUTPUT_DIR / "D4_branch_C_representation_metadata.json"
)
REP_METADATA_PATH.write_text(
    json.dumps(REPRESENTATION_METADATA, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

EXPERIMENT_METADATA = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "branch": BRANCH,
    "branch_name": BRANCH_NAME,
    "parent_branch": PARENT_BRANCH,
    "source_file": SOURCE_PATH.name,
    "source_sha256": SOURCE_SHA256,
    "source_verified": SOURCE_HASH_MATCH,
    "input_representation":
        "Complete deterministically normalised structural Markdown",
    "representation_file": REPRESENTATION_PATH.name,
    "representation_sha256": REPRESENTATION_SHA256,
    "parent_B_equivalence_passed": PARENT_EQUIVALENCE_PASSED,
    "normalisation_integrity_passed":
        normalisation_check["normalisation_integrity_passed"],
    "direct_document_ingestion": False,
    "structural_conversion_applied": True,
    "structural_conversion_inherited_from_branch_B": True,
    "normalisation_applied": True,
    "complete_workbook_retained": True,
    "worksheet_filtering_applied": False,
    "out_of_scope_content_retained": True,
    "fixed_extraction_scope": "Metadata rows 2–9 and 12–86",
    "reference_values_disclosed_to_model": False,
    "reference_values_used_for_transformation": False,
    "expected_record_count_disclosed_to_model": False,
    "manual_response_repair_permitted": False,
    "expected_output_format": "JSON",
    "prompt_file": PROMPT_PATH.name,
    "prompt_sha256": PROMPT_SHA256,
    "execution_environment": "Independent ChatGPT conversation",
    "model": "GPT-5.5",
    "created_at": datetime.now().isoformat(),
    "python_version": sys.version,
    "platform": platform.platform(),
    "validation_status":
        "Pending Stage 4 Branch C validation against the fixed Stage 1 "
        "reference dataset using pre-specified D4 comparison/equivalence rules"
}

METADATA_PATH = OUTPUT_DIR / "D4_branch_C_experiment_metadata.json"
METADATA_PATH.write_text(
    json.dumps(EXPERIMENT_METADATA, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print(json.dumps(EXPERIMENT_METADATA, indent=2, ensure_ascii=False))


{
  "document_id": "D4",
  "document_name": "Eurostat — LFS Metadata Excel — lfsa_esms",
  "branch": "C",
  "branch_name": "Deterministic normalisation",
  "parent_branch": "B",
  "source_file": "D4 - Eurostat – LFS Metadata Excel - lfsa_esms.xlsx",
  "source_sha256": "040f1ace0392f8517fb99d4785b79cf3161cdd8e35f9c0e95bf243fe9baba5e9",
  "source_verified": true,
  "input_representation": "Complete deterministically normalised structural Markdown",
  "representation_file": "D4_branch_C_normalised_markdown.md",
  "representation_sha256": "465f1e7d420803d05bc3a501f6a37968dd2f3eb6df58323138c527f23447956b",
  "parent_B_equivalence_passed": true,
  "normalisation_integrity_passed": true,
  "direct_document_ingestion": false,
  "structural_conversion_applied": true,
  "structural_conversion_inherited_from_branch_B": true,
  "normalisation_applied": true,
  "complete_workbook_retained": true,
  "worksheet_filtering_applied": false,
  "out_of_scope_content_retained": true,
  "fixed_extraction_sc

In [15]:
# ============================================================
# 12. Final pre-extraction control check
# ============================================================
PRECHECK = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,
    "source_identity_verified": SOURCE_HASH_MATCH,
    "parent_B_equivalence_passed": PARENT_EQUIVALENCE_PASSED,
    "normalisation_integrity_passed":
        normalisation_check["normalisation_integrity_passed"],
    "complete_workbook_retained":
        all(worksheet_marker_checks.values()),
    "target_scope_sequence_preserved":
        target_scope_sequence_preserved,
    "representation_exists": REPRESENTATION_PATH.exists(),
    "prompt_exists": PROMPT_PATH.exists(),
    "expected_record_count_disclosed_to_model": False,
    "reference_values_used_for_transformation": False,
    "ready_for_independent_llm_execution": bool(
        SOURCE_HASH_MATCH
        and PARENT_EQUIVALENCE_PASSED
        and normalisation_check["normalisation_integrity_passed"]
        and REPRESENTATION_PATH.exists()
        and PROMPT_PATH.exists()
    )
}

PRECHECK_PATH = OUTPUT_DIR / "D4_branch_C_pre_extraction_check.json"
PRECHECK_PATH.write_text(
    json.dumps(PRECHECK, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print(json.dumps(PRECHECK, indent=2, ensure_ascii=False))

if not PRECHECK["ready_for_independent_llm_execution"]:
    raise ValueError("D4 Branch C is not ready for independent LLM execution.")


{
  "document_id": "D4",
  "branch": "C",
  "source_identity_verified": true,
  "parent_B_equivalence_passed": true,
  "normalisation_integrity_passed": true,
  "complete_workbook_retained": true,
  "target_scope_sequence_preserved": true,
  "representation_exists": true,
  "prompt_exists": true,
  "expected_record_count_disclosed_to_model": false,
  "reference_values_used_for_transformation": false,
  "ready_for_independent_llm_execution": true
}


In [16]:
# ============================================================
# 13. Download pre-extraction Branch C artefacts
# ============================================================
for path in [
    PARENT_CHECK_PATH,
    NORMALISATION_CHECK_PATH,
    REPRESENTATION_PATH,
    PROMPT_PATH,
    REP_METADATA_PATH,
    METADATA_PATH,
    PRECHECK_PATH
]:
    files.download(path)

print(
    "\nIndependent extraction instructions:\n"
    "1. Open a new independent ChatGPT conversation.\n"
    "2. Upload ONLY D4_branch_C_normalised_markdown.md.\n"
    "3. Submit D4_branch_C_prompt.txt exactly once.\n"
    "4. Do not upload the original XLSX, Branch B artefacts, Stage 1 reference values, "
    "or previous extraction outputs.\n"
    "5. Do not manually repair, correct, or regenerate the response.\n"
    "6. Save the complete response exactly as returned in a plain-text file."
)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Independent extraction instructions:
1. Open a new independent ChatGPT conversation.
2. Upload ONLY D4_branch_C_normalised_markdown.md.
3. Submit D4_branch_C_prompt.txt exactly once.
4. Do not upload the original XLSX, Branch B artefacts, Stage 1 reference values, or previous extraction outputs.
5. Do not manually repair, correct, or regenerate the response.
6. Save the complete response exactly as returned in a plain-text file.


In [17]:
# ============================================================
# 14. Upload and preserve the complete raw Branch C response
# ============================================================
uploaded_response = files.upload()

if len(uploaded_response) != 1:
    raise ValueError("Upload exactly one complete raw Branch C response text file.")

RAW_RESPONSE_SOURCE = Path(next(iter(uploaded_response)))
RAW_RESPONSE_TEXT = RAW_RESPONSE_SOURCE.read_text(encoding="utf-8")

RAW_RESPONSE_PATH = OUTPUT_DIR / "D4_branch_C_raw_response.txt"
RAW_RESPONSE_PATH.write_text(RAW_RESPONSE_TEXT, encoding="utf-8")
RAW_RESPONSE_SHA256 = sha256_file(RAW_RESPONSE_PATH)

print("Raw response preserved unchanged.")
print("Raw response SHA-256:", RAW_RESPONSE_SHA256)


Saving D4_branch_C_raw_response.txt to D4_branch_C_raw_response.txt
Raw response preserved unchanged.
Raw response SHA-256: f98b44ab6167ffadc436f7c8bd7583504339869bdda890180d9a395913df028d


In [18]:
# ============================================================
# 15. Parse the raw response without repair
# ============================================================
valid_json = True
json_error = None
raw_extraction = None

try:
    raw_extraction = json.loads(RAW_RESPONSE_TEXT)
except json.JSONDecodeError as exc:
    valid_json = False
    json_error = str(exc)

print("Valid JSON:", valid_json)
if json_error:
    print("JSON parsing error:", json_error)


Valid JSON: True


In [19]:
# ============================================================
# 16. Validate top-level structure, records and field types
# ============================================================
top_level_object_valid = valid_json and isinstance(raw_extraction, dict)
document_id_present = top_level_object_valid and "document_id" in raw_extraction
document_id_correct = document_id_present and raw_extraction.get("document_id") == DOCUMENT_ID
branch_present = top_level_object_valid and "branch" in raw_extraction
branch_correct = branch_present and raw_extraction.get("branch") == BRANCH
records_present = top_level_object_valid and "records" in raw_extraction
records_is_list = records_present and isinstance(raw_extraction.get("records"), list)

records = raw_extraction["records"] if records_is_list else []

record_structure_issues = []
field_type_issues = []
publication_flag_issues = []

for record_index, record in enumerate(records):
    if not isinstance(record, dict):
        record_structure_issues.append({
            "record_index": record_index,
            "issue": "Record is not a JSON object"
        })
        continue

    actual_fields = set(record.keys())
    expected_fields = set(EXPECTED_FIELDS)

    missing_fields = sorted(expected_fields - actual_fields)
    extra_fields = sorted(actual_fields - expected_fields)

    if missing_fields or extra_fields:
        record_structure_issues.append({
            "record_index": record_index,
            "missing_fields": missing_fields,
            "extra_fields": extra_fields
        })

    for field in EXPECTED_FIELDS:
        value = record.get(field)
        if value is not None and not isinstance(value, str):
            field_type_issues.append({
                "record_index": record_index,
                "field": field,
                "observed_type": type(value).__name__
            })

    observed_flag = record.get("Publication Restricted")
    if observed_flag not in ALLOWED_PUBLICATION_FLAGS:
        publication_flag_issues.append({
            "record_index": record_index,
            "observed_flag": observed_flag
        })

records_with_type_issues = len({
    issue["record_index"]
    for issue in field_type_issues
})

print("Top-level object valid:", top_level_object_valid)
print("Records:", len(records))
print("Record structure issues:", len(record_structure_issues))
print("Records with type issues:", records_with_type_issues)
print("Publication flag issues:", len(publication_flag_issues))


Top-level object valid: True
Records: 83
Record structure issues: 0
Records with type issues: 0
Publication flag issues: 0


In [20]:
# ============================================================
# 17. Scope and representation-adherence diagnostics
# ============================================================
record_count = len(records)

header_record_count = sum(
    1
    for record in records
    if isinstance(record, dict) and record.get("Section") == "Header"
)

concept_record_count = record_count - header_record_count

record_count_valid = record_count == EXPECTED_RECORD_COUNT
header_count_valid = header_record_count == EXPECTED_HEADER_RECORD_COUNT
concept_count_valid = concept_record_count == EXPECTED_CONCEPT_RECORD_COUNT

scope_complete = bool(
    record_count_valid
    and header_count_valid
    and concept_count_valid
)

record_keys = [
    (record.get("Section"), record.get("Concept Name"))
    for record in records
    if isinstance(record, dict)
]

duplicate_record_key_count = len(record_keys) - len(set(record_keys))

missing_values_by_field = {
    field: sum(
        1
        for record in records
        if (
            not isinstance(record, dict)
            or field not in record
            or record.get(field) is None
        )
    )
    for field in EXPECTED_FIELDS
}

unexpected_source_row_fields = sum(
    1
    for record in records
    if isinstance(record, dict) and "Source Row" in record
)

extracted_values_with_html_tags = []
extracted_values_with_html_entities = []

for record_index, record in enumerate(records):
    if not isinstance(record, dict):
        continue

    value = record.get("Concept Value")
    if not isinstance(value, str):
        continue

    if HTML_TAG_PATTERN.search(value):
        extracted_values_with_html_tags.append(record_index)

    if HTML_ENTITY_PATTERN.search(value):
        extracted_values_with_html_entities.append(record_index)

html_reintroduction_detected = bool(extracted_values_with_html_tags)
html_entity_reintroduction_detected = bool(extracted_values_with_html_entities)

representation_adherence_valid = bool(
    unexpected_source_row_fields == 0
    and not html_reintroduction_detected
    and not html_entity_reintroduction_detected
)

print("Expected records:", EXPECTED_RECORD_COUNT)
print("Observed records:", record_count)
print("Scope complete:", scope_complete)
print("Duplicate keys:", duplicate_record_key_count)
print("HTML reintroduced:", html_reintroduction_detected)
print("HTML entities reintroduced:", html_entity_reintroduction_detected)


Expected records: 83
Observed records: 83
Scope complete: True
Duplicate keys: 0
HTML reintroduced: False
HTML entities reintroduced: False


In [21]:
# ============================================================
# 18. Keep schema validity separate from scope completeness
# ============================================================
schema_validity = bool(
    valid_json
    and top_level_object_valid
    and document_id_correct
    and branch_correct
    and records_is_list
    and len(record_structure_issues) == 0
    and records_with_type_issues == 0
    and len(publication_flag_issues) == 0
)

STRUCTURE_CHECK = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,
    "valid_json": bool(valid_json),
    "json_error": json_error,
    "top_level_object_valid": bool(top_level_object_valid),
    "document_id_present": bool(document_id_present),
    "document_id_correct": bool(document_id_correct),
    "branch_present": bool(branch_present),
    "branch_correct": bool(branch_correct),
    "records_present": bool(records_present),
    "records_is_list": bool(records_is_list),
    "schema_validity": bool(schema_validity),

    "expected_record_count": EXPECTED_RECORD_COUNT,
    "observed_record_count": int(record_count),
    "record_count_valid": bool(record_count_valid),

    "expected_header_record_count": EXPECTED_HEADER_RECORD_COUNT,
    "observed_header_record_count": int(header_record_count),
    "header_count_valid": bool(header_count_valid),

    "expected_concept_record_count": EXPECTED_CONCEPT_RECORD_COUNT,
    "observed_concept_record_count": int(concept_record_count),
    "concept_count_valid": bool(concept_count_valid),

    "scope_complete": bool(scope_complete),

    "records_with_structure_issues": len(record_structure_issues),
    "record_structure_issues": record_structure_issues,

    "records_with_type_issues": records_with_type_issues,
    "field_type_issues": field_type_issues,

    "publication_flag_issue_count": len(publication_flag_issues),
    "publication_flag_issues": publication_flag_issues,

    "duplicate_record_key_count": int(duplicate_record_key_count),
    "missing_values_by_field": missing_values_by_field,

    "unexpected_source_row_field_count": int(unexpected_source_row_fields),

    "html_reintroduction_detected": bool(html_reintroduction_detected),
    "html_entity_reintroduction_detected":
        bool(html_entity_reintroduction_detected),

    "representation_adherence_valid":
        bool(representation_adherence_valid)
}

STRUCTURE_CHECK_PATH = OUTPUT_DIR / "D4_branch_C_structure_check.json"
STRUCTURE_CHECK_PATH.write_text(
    json.dumps(STRUCTURE_CHECK, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print(json.dumps(STRUCTURE_CHECK, indent=2, ensure_ascii=False))


{
  "document_id": "D4",
  "branch": "C",
  "valid_json": true,
  "json_error": null,
  "top_level_object_valid": true,
  "document_id_present": true,
  "document_id_correct": true,
  "branch_present": true,
  "branch_correct": true,
  "records_present": true,
  "records_is_list": true,
  "schema_validity": true,
  "expected_record_count": 83,
  "observed_record_count": 83,
  "record_count_valid": true,
  "expected_header_record_count": 8,
  "observed_header_record_count": 8,
  "header_count_valid": true,
  "expected_concept_record_count": 75,
  "observed_concept_record_count": 75,
  "concept_count_valid": true,
  "scope_complete": true,
  "records_with_structure_issues": 0,
  "record_structure_issues": [],
  "records_with_type_issues": 0,
  "field_type_issues": [],
  "publication_flag_issue_count": 0,
  "publication_flag_issues": [],
  "duplicate_record_key_count": 0,
  "missing_values_by_field": {
    "Section": 0,
    "Concept Name": 0,
    "Concept Value": 15,
    "Publication Rest

In [22]:
# ============================================================
# 19. Preserve parsed extraction only when JSON is valid
# ============================================================
PARSED_EXTRACTION_PATH = (
    OUTPUT_DIR / "D4_branch_C_parsed_extraction.json"
)

if valid_json:
    PARSED_EXTRACTION_PATH.write_text(
        json.dumps(raw_extraction, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    print("Parsed extraction saved:", PARSED_EXTRACTION_PATH.name)
else:
    print(
        "No parsed extraction created because the preserved raw response is invalid JSON."
    )


Parsed extraction saved: D4_branch_C_parsed_extraction.json


In [23]:
# ============================================================
# 20. Create final Branch C experiment summary
# ============================================================
EXPERIMENT_SUMMARY = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "branch": BRANCH,
    "branch_name": BRANCH_NAME,
    "parent_branch": PARENT_BRANCH,

    "source_file": SOURCE_PATH.name,
    "source_sha256": SOURCE_SHA256,
    "source_verified": SOURCE_HASH_MATCH,

    "input_representation":
        "Complete deterministically normalised structural Markdown",
    "representation_file": REPRESENTATION_PATH.name,
    "representation_sha256": REPRESENTATION_SHA256,

    "parent_B_equivalence_passed": PARENT_EQUIVALENCE_PASSED,
    "normalisation_integrity_passed":
        normalisation_check["normalisation_integrity_passed"],

    "structural_conversion_inherited_from_branch_B": True,
    "normalisation_applied": True,
    "complete_workbook_retained": True,
    "worksheet_filtering_applied": False,
    "out_of_scope_content_retained": True,

    "reference_values_used_for_transformation": False,
    "expected_record_count_disclosed_to_model": False,

    "raw_response_preserved": True,
    "raw_response_sha256": RAW_RESPONSE_SHA256,

    "valid_json": bool(valid_json),
    "schema_validity": bool(schema_validity),

    "expected_record_count": EXPECTED_RECORD_COUNT,
    "observed_record_count": int(record_count),
    "scope_complete": bool(scope_complete),

    "representation_adherence_valid": bool(representation_adherence_valid),

    "records_with_structure_issues": len(record_structure_issues),
    "records_with_type_issues": records_with_type_issues,
    "publication_flag_issue_count": len(publication_flag_issues),
    "duplicate_record_key_count": int(duplicate_record_key_count),

    "parsed_extraction_created": bool(valid_json),
    "accuracy_validation_completed": False,

    "validation_status": (
        "Pending Stage 4 Branch C validation against the fixed Stage 1 "
        "reference dataset using pre-specified D4 comparison/equivalence rules"
        if valid_json
        else
        "Not content-evaluable because the preserved raw Branch C response "
        "is invalid JSON"
    )
}

SUMMARY_PATH = OUTPUT_DIR / "D4_branch_C_experiment_summary.json"
SUMMARY_PATH.write_text(
    json.dumps(EXPERIMENT_SUMMARY, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print(json.dumps(EXPERIMENT_SUMMARY, indent=2, ensure_ascii=False))


{
  "document_id": "D4",
  "document_name": "Eurostat — LFS Metadata Excel — lfsa_esms",
  "branch": "C",
  "branch_name": "Deterministic normalisation",
  "parent_branch": "B",
  "source_file": "D4 - Eurostat – LFS Metadata Excel - lfsa_esms.xlsx",
  "source_sha256": "040f1ace0392f8517fb99d4785b79cf3161cdd8e35f9c0e95bf243fe9baba5e9",
  "source_verified": true,
  "input_representation": "Complete deterministically normalised structural Markdown",
  "representation_file": "D4_branch_C_normalised_markdown.md",
  "representation_sha256": "465f1e7d420803d05bc3a501f6a37968dd2f3eb6df58323138c527f23447956b",
  "parent_B_equivalence_passed": true,
  "normalisation_integrity_passed": true,
  "structural_conversion_inherited_from_branch_B": true,
  "normalisation_applied": true,
  "complete_workbook_retained": true,
  "worksheet_filtering_applied": false,
  "out_of_scope_content_retained": true,
  "reference_values_used_for_transformation": false,
  "expected_record_count_disclosed_to_model": fa

In [24]:
# ============================================================
# 21. Final artefact inventory and download
# ============================================================
artefacts = [
    PARENT_CHECK_PATH,
    NORMALISATION_CHECK_PATH,
    REPRESENTATION_PATH,
    PROMPT_PATH,
    REP_METADATA_PATH,
    METADATA_PATH,
    PRECHECK_PATH,
    RAW_RESPONSE_PATH,
    STRUCTURE_CHECK_PATH,
    SUMMARY_PATH
]

if valid_json:
    artefacts.append(PARSED_EXTRACTION_PATH)

print("Final D4 Branch C artefacts:")
for path in artefacts:
    print("-", path.name, "| exists:", path.exists())

for path in artefacts:
    if path.exists():
        files.download(path)


Final D4 Branch C artefacts:
- D4_branch_C_parent_B_equivalence_check.json | exists: True
- D4_branch_C_normalisation_check.json | exists: True
- D4_branch_C_normalised_markdown.md | exists: True
- D4_branch_C_prompt.txt | exists: True
- D4_branch_C_representation_metadata.json | exists: True
- D4_branch_C_experiment_metadata.json | exists: True
- D4_branch_C_pre_extraction_check.json | exists: True
- D4_branch_C_raw_response.txt | exists: True
- D4_branch_C_structure_check.json | exists: True
- D4_branch_C_experiment_summary.json | exists: True
- D4_branch_C_parsed_extraction.json | exists: True


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>